In [5]:
!pip install -q transformers torch

In [9]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForQuestionAnswering,
    pipeline
)
import torch

# ============================================================
# TEXT SUMMARIZATION USING BART
# ============================================================

model_name = "facebook/bart-large-cnn"

tokenizer_summary = AutoTokenizer.from_pretrained(model_name)
model_summary = AutoModelForSeq2SeqLM.from_pretrained(model_name)

article = """Generative AI refers to a class of artificial intelligence models capable of
producing new content such as text, images, audio, and video. Large Language Models (LLMs)
such as GPT and LLaMA are trained on massive text corpora and can perform a wide range of
natural language tasks including translation, summarization, and question answering. These
models are increasingly being deployed in industry applications ranging from customer support
to software development, transforming how humans interact with machines."""

inputs_summary = tokenizer_summary(
    article,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

summary_ids = model_summary.generate(
    **inputs_summary,
    max_length=45,
    min_length=20,
    do_sample=False
)

summary = tokenizer_summary.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("========== TEXT SUMMARIZATION ==========")
print("Original Text:\n", article)
print("\nSummary:\n", summary)


# ============================================================
# QUESTION ANSWERING USING DISTILBERT (Manual Implementation)
# ============================================================

qa_model_name = "distilbert-base-cased-distilled-squad"

tokenizer_qa = AutoTokenizer.from_pretrained(qa_model_name)
model_qa = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)

question = "What are Large Language Models trained on?"

inputs_qa = tokenizer_qa(
    question,
    article,
    add_special_tokens=True,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model_qa(**inputs_qa)

answer_start_scores = outputs.start_logits
answer_end_scores = outputs.end_logits

# Get the most likely beginning and end of the answer span
answer_start = torch.argmax(answer_start_scores)
answer_end = torch.argmax(answer_end_scores) + 1  # Add 1 to make it exclusive

# Convert tokens to words
input_ids = inputs_qa["input_ids"].tolist()[0]
answer_tokens = input_ids[answer_start:answer_end]
answer = tokenizer_qa.decode(answer_tokens)

# Calculate confidence (using max score for simplicity, for proper confidence, more complex methods are needed)
confidence = torch.max(torch.softmax(answer_start_scores, dim=1)) * torch.max(torch.softmax(answer_end_scores, dim=1))

print("\n========== QUESTION ANSWERING ==========")
print("Question:", question)
print("Answer:", answer)
print("Confidence:", round(confidence.item(), 3))


# ============================================================
# RESULT
# ============================================================

print("\n========== RESULT ==========")
print("A text summarization system using BART and a question-answering")
print("system using DistilBERT-SQuAD were successfully developed and tested.")

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

========== TEXT SUMMARIZATION ==========
Original Text:
 Generative AI refers to a class of artificial intelligence models capable of
producing new content such as text, images, audio, and video. Large Language Models (LLMs)
such as GPT and LLaMA are trained on massive text corpora and can perform a wide range of
natural language tasks including translation, summarization, and question answering. These
models are increasingly being deployed in industry applications ranging from customer support
to software development, transforming how humans interact with machines.

Summary:
 Large Language Models (LLMs) are trained on massive text corpora. They can perform a wide range of natural language tasks including translation, summarization, and question answering.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


========== QUESTION ANSWERING ==========
Question: What are Large Language Models trained on?
Answer: massive text corpora
Confidence: 0.892

========== RESULT ==========
A text summarization system using BART and a question-answering
system using DistilBERT-SQuAD were successfully developed and tested.
